# Why Is a GPU Faster Than a CPU?

## 1. Core Idea: Latency vs. Throughput

**CPU is optimized for low latency; GPU is optimized for high throughput.**

- **CPU:** Fewer, more sophisticated cores. Designed to execute complex, sequential tasks quickly.
- **GPU:** Many parallel execution units. Designed to perform massive numbers of similar operations simultaneously.

**Example:**

Consider two tasks, each requiring 1 million additions:

- **Task A:** All additions are independent → GPU can execute many simultaneously.
- **Task B:** Each addition depends on the previous result → Limited parallelism; GPU loses much of its advantage.

**Key insight:** The number of FLOPs alone does not determine execution time. Available parallelism matters.

## 2. FLOPs vs. FP32

- **FLOP:** One floating-point operation.
- **FLOPS:** Floating-point operations per second (compute throughput).
- **FP32:** A floating-point number represented using 32 bits (4 bytes).

**Example:**

`c = a * b + d` → 2 FLOPs, whether using FP32 or FP16.

Different precisions can have very different hardware throughput.

## 3. Arithmetic Intensity: Computation vs. Data Movement

Moving data costs time. Ideally, we want to perform many calculations on data already loaded into fast memory.

$$
I=\frac{\text{FLOPs}}{\text{Bytes transferred}}
$$

**Example A: Vector Addition**

`C[i] = A[i] + B[i]`

Using FP32:

- Read A and B: 8 bytes
- Write C: 4 bytes
- Compute: 1 FLOP

$$
I=\frac{1}{12}\text{ FLOPs/Byte}
$$

Increasing vector length does not increase arithmetic intensity.

**Example B: Matrix Multiplication**

`C = A @ B`, with three $N\times N$ matrices.

- Computation: approximately $2N^3$ FLOPs.
- Ideal data movement: $12N^2$ bytes.

$$
I_{\text{ideal}}\approx\frac{2N^3}{12N^2}=\frac N6
$$

**Key insight:** Matrix multiplication reuses the same elements many times. As matrix size increases, its *ideal* arithmetic intensity increases.

Actual memory traffic depends on cache capacity and implementation.

## 4. Roofline Model: Two Fundamental Bottlenecks

Performance is limited by:

1. **Compute throughput:** How many operations can the processor execute per second?
2. **Memory bandwidth:** How quickly can data reach the processor?

$$
P_{\text{achievable}}\leq\min(P_{\text{peak}},B\times I)
$$

Where:

- $P_{\text{peak}}$: Peak compute throughput.
- $B$: Memory bandwidth.
- $I$: Arithmetic intensity.

**Example:**

A GPU has:

- 100 TFLOPS peak compute throughput.
- 1 TB/s memory bandwidth.

For vector addition, $I=1/12$:

$$
P_{\text{achievable}}\leq83.3\text{ GFLOPS}
$$

Despite having 100 TFLOPS of theoretical compute capability, the GPU cannot fully utilize it for this memory-bound task.

**Key insight:** More compute units are useless if data cannot arrive fast enough.

## 5. Tiling: Maximizing Data Reuse

Both CPUs and GPUs use **tiling (blocking)** to reduce expensive memory accesses.

**Example:**

In matrix multiplication, one element of A may be used in $N$ different multiplications.

Instead of repeatedly reading it from DRAM:

1. Load a small matrix tile into fast memory.
2. Reuse its elements for many calculations.
3. Move to the next tile.

**Key insight:** Tiling improves arithmetic intensity by reducing data movement, but it does not inherently make GPUs faster than CPUs. Both can use it.

## 6. GPU's Architectural Advantages

Even if CPU and GPU achieve the same arithmetic intensity, GPUs often have:

- **More parallel arithmetic units:** Higher total compute throughput.
- **Higher memory bandwidth:** More data delivered per second.
- **Latency hiding:** When one group of threads waits for memory, another ready group can execute.
- **Specialized hardware:** Tensor Cores accelerate supported matrix operations.

**Example:**

Suppose both CPU and GPU execute a compute-bound task requiring $10^{12}$ FLOPs.

- CPU: 1 TFLOPS → 1 second.
- GPU: 20 TFLOPS → 0.05 seconds.

The GPU is theoretically 20× faster, assuming sufficient parallelism and utilization.

## 7. Final Mental Model

**GPU performance depends on three fundamental factors:**

1. **Parallelism:** How much work can execute simultaneously?
2. **Compute throughput:** How many operations can hardware perform per second?
3. **Memory bandwidth and reuse:** How efficiently can data feed those operations?

The Roofline Model captures compute and memory limits. Parallelism, dependencies, and execution overhead determine how closely real programs approach those limits.

**The ultimate insight:**

> CPU architecture prioritizes finishing individual tasks quickly. GPU architecture prioritizes completing massive amounts of parallel work efficiently.
>
> GPUs are fast not simply because they have more cores, but because their entire architecture is designed for throughput.